# Dynamic Peer-to-Peer Swarm Multi-Agent Memory Evaluation

## The Problem

In a *swarm* peer-to-peer system, no predetermined sequence exists. Each peer decides whether to hand off, and to whom, based on its own reasoning. This makes context propagation failures even harder to detect: not only can a peer inherit stale memory, but the entire execution path can diverge depending on which handoffs get called. When the brief changes after the first run, peers must reconcile both the new instructions AND the non-deterministic handoff graph from the previous run.

## What This Notebook Does

We run a **restaurant industry market research** dynamic swarm through two sessions:

1. **Simple Session** — One run with a stable research brief. Peers decide handoffs dynamically via tool calls.
2. **Session with Feedback** — First run with baseline brief. Then scope expands (US → North America) and the swarm runs again. Shared memory from run 1 persists — tests whether peers choose different handoffs and reconcile stale analysis.

## Architecture

```
              ┌──────────────────────────┐
              │   Research Brief (User)   │
              └────────────┬─────────────┘
                           │
                           ▼
              ┌──────────────────────────┐
              │  market_trends_analyst    │  Can call:
              │                          │   handoff_to_customer_insights
              └────────────┬─────────────┘
                           │ (LLM chooses)
                           ▼
              ┌──────────────────────────┐
              │ customer_insights_analyst │  Can call:
              │                          │   handoff_to_strategy_synth
              └────────────┬─────────────┘   handoff_to_market_trends
                           │ (LLM chooses)
                           ▼
              ┌──────────────────────────┐
              │   strategy_synthesizer   │  Terminal peer (no handoff tools)
              └──────────────────────────┘
                           │
              ┌────────────▼─────────────┐
              │  Shared Memory (list)    │
              │  All peers share it      │
              └──────────────────────────┘
```

**Flow:**
- Each peer has `handoff_to_<agent>` tools — calling one signals the next peer to dispatch
- A dispatcher loop runs the current peer, checks which handoff (if any) was called, routes accordingly
- If a peer responds without calling a handoff tool, the swarm ends
- Order is emergent — peer 2 could hand back to peer 1 if it needs more analysis
- Same `ListMemoryHook` pattern as the hub-spoke notebook: each peer reads memory on init, writes on message

## Metrics Evaluated

**Memory Context Metrics** (LLM-as-judge, 1-5 scale):

| Metric | What it measures |
|--------|------------------|
| Context Freshness | Is the peer working with the latest information? |
| Handoff Completeness | Did the handoff include all facts the peer needs? |
| Context Utilization | Did the peer use the context it read from memory? |
| State Consistency | Do all peers agree on key facts? |
| Memory Write Accuracy | Is what the peer wrote to memory factually correct? |
| Redundant Context | How much repeated/irrelevant context was transferred? |
| C2 Alignment | Cosine similarity of peer outputs (Titan embeddings) |

**Memory Latency Metrics** (timers + token counts):

| Metric | What it measures |
|--------|------------------|
| Memory Read/Write Latency | Time for memory operations |
| Coordination Latency % | Fraction of time spent on coordination vs reasoning |
| Coordination Token % | Fraction of tokens spent on context vs generation |

## Step 1 — Setup

In [ ]:
!pip install -qr requirements.txt

In [ ]:
import os
import time
import logging
from IPython.display import display, Markdown

from strands import Agent, tool
from strands.hooks import AgentInitializedEvent, HookProvider, HookRegistry, MessageAddedEvent

from metrics_collector import MetricsCollector
from eval_helpers import (
    format_memory, print_memory,
    compute_embeddings, c2_alignment_report,
)
from model_config import (
    AGENT_MODEL_ID,
    MARKET_TRENDS_PROMPT, CUSTOMER_INSIGHTS_PROMPT, STRATEGY_SYNTH_PROMPT,
)

region = os.getenv("AWS_REGION", "us-west-2")

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s",
                    datefmt="%Y-%m-%d %H:%M:%S")
logger = logging.getLogger("dynamic-swarm")

## Step 2 — Shared Memory (Python list)

Same pattern as the hub-and-spoke notebook. A Python list that all peers read from and append to. Used by the `ListMemoryHook` below.

In [ ]:
# format_memory() is imported from eval_helpers — same function as the other notebooks
print(format_memory([
    {"agent": "example", "role": "assistant", "content": "Shared memory entry."},
]))

## Step 3 — Instrumented Memory Hook

Identical to the hub-spoke notebook. Each peer gets this hook attached:
- `on_agent_initialized` — read shared memory, inject into system prompt, record context
- `on_message_added` — if the peer produced text (not just tool calls), append to shared memory

We filter out tool-call-only messages — when a peer calls a handoff tool, the message has `toolUse` content, not text.

In [ ]:
class ListMemoryHook(HookProvider):
    """Reads/writes a shared Python list for each peer agent."""

    def __init__(self, memory: list, collector: MetricsCollector, agent_name: str):
        self.memory = memory
        self.collector = collector
        self.agent_name = agent_name

    def on_agent_initialized(self, event: AgentInitializedEvent):
        t0 = time.perf_counter()
        context = format_memory(self.memory)
        self.collector.record_memory_read_latency(self.agent_name, time.perf_counter() - t0)

        if context:
            event.agent.system_prompt += (
                f"\n\nShared memory from other agents:\n{context}"
                "\n\nUse this context. Reference specific details from other agents.")
            logger.info(f"[{self.agent_name}] Read {len(self.memory)} entries from memory")

        self.collector.record_retrieved_context(self.agent_name, context)

    def on_message_added(self, event: MessageAddedEvent):
        last = event.agent.messages[-1]
        if last["role"] != "assistant":
            return

        # Extract text content (skip tool-use-only messages)
        text_parts = []
        for block in last.get("content", []):
            if isinstance(block, dict) and block.get("text"):
                text_parts.append(block["text"])
        if not text_parts:
            return

        response_text = "\n".join(text_parts)
        self.collector.record_response(self.agent_name, response_text)

        t0 = time.perf_counter()
        self.memory.append({
            "agent": self.agent_name,
            "role": "assistant",
            "content": response_text,
            "ts": time.time(),
        })
        self.collector.record_memory_write_latency(self.agent_name, time.perf_counter() - t0)

    def register_hooks(self, registry: HookRegistry):
        registry.add_callback(AgentInitializedEvent, self.on_agent_initialized)
        registry.add_callback(MessageAddedEvent, self.on_message_added)

## Step 4 — Handoff Tools and Dispatcher

Each peer can call one of these handoff tools to pass control to another peer. The dispatcher reads which tool was called (via a shared `_handoff_target` dict) and routes accordingly.

If a peer responds without calling any handoff tool, the swarm ends.

In [ ]:
# Shared dispatcher state — set by handoff tools, read by the dispatcher loop
_handoff_target = {"next": None}

@tool
def handoff_to_customer_insights(reason: str) -> str:
    """Hand off to the customer insights analyst.

    Args:
        reason: Why you're handing off (1 sentence)
    Returns:
        Confirmation that the handoff was queued
    """
    _handoff_target["next"] = "customer_insights"
    return f"Handoff queued: {reason}"

@tool
def handoff_to_strategy_synth(reason: str) -> str:
    """Hand off to the strategy synthesizer.

    Args:
        reason: Why you're handing off (1 sentence)
    Returns:
        Confirmation that the handoff was queued
    """
    _handoff_target["next"] = "strategy_synth"
    return f"Handoff queued: {reason}"

@tool
def handoff_to_market_trends(reason: str) -> str:
    """Hand off back to market trends analyst if more analysis is needed.

    Args:
        reason: Why more market analysis is needed
    Returns:
        Confirmation that the handoff was queued
    """
    _handoff_target["next"] = "market_trends"
    return f"Handoff queued: {reason}"

In [ ]:
# Peer specs: {name: (base_prompt, [allowed handoff targets])}
PEER_SPECS = {
    "market_trends": (MARKET_TRENDS_PROMPT, ["customer_insights"]),
    "customer_insights": (CUSTOMER_INSIGHTS_PROMPT, ["strategy_synth", "market_trends"]),
    "strategy_synth": (STRATEGY_SYNTH_PROMPT, []),  # terminal peer
}

HANDOFF_TOOL_MAP = {
    "market_trends": handoff_to_market_trends,
    "customer_insights": handoff_to_customer_insights,
    "strategy_synth": handoff_to_strategy_synth,
}


def build_peer(name: str, memory: list, collector: MetricsCollector) -> Agent:
    """Create a peer agent with its handoff tools and memory hook attached."""
    base_prompt, allowed = PEER_SPECS[name]
    prompt = base_prompt + (
        f"\n\nYou are the '{name}' peer in a multi-agent swarm. Complete your analysis, "
        f"then decide: if another peer should continue the work, call one of your handoff "
        f"tools. If the analysis is complete and no further work is needed, respond with "
        f"your final answer WITHOUT calling any handoff tool — that ends the swarm.")
    tools = [HANDOFF_TOOL_MAP[target] for target in allowed]
    hook = ListMemoryHook(memory, collector, name)
    return Agent(name=name, model=AGENT_MODEL_ID, system_prompt=prompt,
                 tools=tools, hooks=[hook])

## Step 5 — Session Runner

The dispatcher loop: start with the first peer, run it, check what handoff (if any) was called, repeat until a peer doesn't hand off. Emits metrics via the collector at each step.

In [ ]:
def run_dynamic_swarm(task: str, shared_memory: list, collector: MetricsCollector,
                     turn_number: int, session_label: str, max_iterations: int = 6) -> dict:
    """Run the swarm starting with market_trends. Follows dynamic handoffs."""
    collector.begin_turn(turn_number, task)
    responses = {}
    current = next(iter(PEER_SPECS.keys()))  # start with market_trends
    iterations = 0

    print(f"\n{'='*60}")
    print(f"[{session_label}] Turn {turn_number}: {task[:80]}...")
    print(f"{'='*60}")

    while current and iterations < max_iterations:
        iterations += 1
        _handoff_target["next"] = None  # reset before each peer

        print(f"\n-- Iteration {iterations}: running {current} --")
        collector.record_handoff(current, task)

        agent = build_peer(current, shared_memory, collector)
        t0 = time.perf_counter()
        resp = agent(task)
        latency = time.perf_counter() - t0

        collector.record_agent_latency(current, latency)
        usage = getattr(resp, "usage", None) or {}
        collector.record_token_usage(current,
            usage.get("inputTokens", 0), usage.get("outputTokens", 0))

        responses[current] = str(resp)
        print(f"  {current}: {str(resp)[:200]}...")

        next_peer = _handoff_target["next"]
        if next_peer and next_peer in PEER_SPECS:
            print(f"  → {current} handed off to {next_peer}")
            current = next_peer
        else:
            print(f"  → {current} produced final answer, swarm ends")
            current = None

    collector.end_turn()
    return responses


def run_session(conversation: list, session_label: str) -> tuple:
    """Run one or more turns sharing the same memory."""
    shared_memory = []
    collector = MetricsCollector(region=region)
    for i, task in enumerate(conversation, start=1):
        run_dynamic_swarm(task, shared_memory, collector,
                          turn_number=i, session_label=session_label)
    return collector, shared_memory

## Step 6 — Simple Session

One run with a stable research brief. Peers decide handoffs as they go.

In [ ]:
simple_conversation = [
    "Analyse the fast-casual restaurant market in the United States. "
    "Key players include Chipotle, Panera Bread, and Sweetgreen. "
    "Target segment: health-conscious urban diners aged 25-45. "
    "Market size: $60 billion, growing at 8% annually.",
]

simple_metrics, simple_memory = run_session(simple_conversation, "simple")

### Simple Session — Context Flow Trace

In [ ]:
display(Markdown(simple_metrics.trace_report()))

### Simple Session — Shared Memory Contents

In [ ]:
print_memory(simple_memory)

## Step 7 — Session with Feedback

First run with the baseline brief. Then the user expands the scope (US → North America) and the swarm runs again with shared memory still holding run 1's analysis. Tests whether peers reconcile or inherit stale context.

In [ ]:
feedback_conversation = [
    "Analyse the fast-casual restaurant market in the United States. "
    "Key players include Chipotle, Panera Bread, and Sweetgreen. "
    "Target segment: health-conscious urban diners aged 25-45. "
    "Market size: $60 billion, growing at 8% annually.",

    "Actually, expand the analysis to cover all of North America, not just the United States. "
    "Include Canadian and Mexican fast-casual chains as well. "
    "The North American market size is $85 billion. "
    "Keep the same focus on health-conscious urban diners aged 25-45.",
]

feedback_metrics, feedback_memory = run_session(feedback_conversation, "feedback")

### Feedback Session — Context Flow Trace

In [ ]:
display(Markdown(feedback_metrics.trace_report()))

### Feedback Session — Shared Memory Contents

In [ ]:
print_memory(feedback_memory)

## Step 8 — Run LLM-as-Judge Evaluations

In [ ]:
print("Evaluating simple session...")
simple_metrics.evaluate_all()

print("Evaluating feedback session...")
feedback_metrics.evaluate_all()

### C2 Alignment (Titan embeddings)

Cosine similarity between peer outputs. Low similarity = peers diverged in their understanding.

In [ ]:
# Build {agent_name: last response} from each session's memory
simple_final = {}
for e in simple_memory:
    simple_final[e["agent"]] = e["content"]  # keeps the LAST entry per agent

feedback_final = {}
for e in feedback_memory:
    feedback_final[e["agent"]] = e["content"]

simple_embeddings = compute_embeddings(simple_final, region=region)
feedback_embeddings = compute_embeddings(feedback_final, region=region)

## Step 9 — Results: Simple Session

In [ ]:
display(Markdown("## Simple Session\n\n" + simple_metrics.context_metrics_report()))
display(Markdown(simple_metrics.latency_metrics_report()))
display(Markdown(c2_alignment_report(simple_embeddings, "Simple Session")))

## Step 10 — Results: Session with Feedback

In [ ]:
display(Markdown("## Session with Feedback\n\n" + feedback_metrics.context_metrics_report()))
display(Markdown(feedback_metrics.latency_metrics_report()))
display(Markdown(c2_alignment_report(feedback_embeddings, "Feedback Session")))

## Step 11 — Side-by-Side Comparison

In [ ]:
display(Markdown(MetricsCollector.comparison_report(
    simple_metrics, feedback_metrics,
    label_a="Simple Session", label_b="Session with Feedback")))

## Visual Analysis

In [ ]:
import matplotlib.pyplot as plt
from eval_helpers import (
    plot_context_metrics_radar, plot_latency_breakdown,
    plot_session_comparison, plot_c2_heatmap
)

In [ ]:
fig = plot_context_metrics_radar(simple_metrics, "Simple Session")
plt.show()

fig = plot_context_metrics_radar(feedback_metrics, "Feedback Session")
plt.show()

In [ ]:
fig = plot_latency_breakdown(simple_metrics, "Simple Session")
plt.show()

fig = plot_latency_breakdown(feedback_metrics, "Feedback Session")
plt.show()

In [ ]:
fig = plot_session_comparison(simple_metrics, feedback_metrics,
                              "Simple Session", "Feedback Session")
plt.show()

In [ ]:
if simple_embeddings:
    fig = plot_c2_heatmap(simple_embeddings, "Simple Session")
    plt.show()

if feedback_embeddings:
    fig = plot_c2_heatmap(feedback_embeddings, "Feedback Session")
    plt.show()